<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c40-header.png" alt="Nextia Learning · Experiments and Causal Impact: Did Our Change Work?" width="100%">

# Solution: Get more precise with CUPED

**[Get more precise with CUPED](https://learning.nextia-ai.com/courses/experiments/m03/get-more-precise-with-cuped/)** · Experiments and Causal Impact: Did Our Change Work? · Module 3, lesson 3 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** compute CUPED by hand: theta, the adjusted spending, the narrower interval and the share of the variance it removes, and to see why the covariate must come from before the test.

| | |
|---|---|
| **Time** | About 25 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | Welch's t-test and the interval, from [The result and its uncertainty](https://learning.nextia-ai.com/courses/experiments/m03/the-result-and-its-uncertainty/); variance, from the [mathematics course](https://learning.nextia-ai.com/courses/math/m01/describe-a-dataset/). |
| **Tested** | Python 3.12 with pandas 3.1.1, SciPy 1.15.3, statsmodels 0.15.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/experiments/m03/get-more-precise-with-cuped/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C40/M03-L03-get-more-precise-with-cuped/get-more-precise-with-cuped-solution.ipynb).

## Setup: install the packages

Run the next cell first. It installs the versions that the course uses: pandas 3.1.1, SciPy 1.15.3 and statsmodels 0.15.0, and matplotlib 3.11.2 for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with these versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 "scipy==1.15.3; python_version < '3.14'" "scipy==1.18.1; python_version >= '3.14'" statsmodels==0.15.0 matplotlib==3.11.2
import importlib.metadata as _md
for _name in ["pandas", "scipy", "statsmodels", "matplotlib"]:
    print(_name, _md.version(_name))

## Setup: get the practice data

Run the next cell. It downloads the simulator's data for this lesson (less than 3 MB) into a folder `larkfield-experiments/data`, and checks each file's checksum. It needs an internet connection and no account.

You should see `Ready: 2 files in data/.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-experiments` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: the course simulator made it. It has no real people. Each folder has `truth.json`: the true effects that the simulator knows. A real test never has this file.

In [ ]:
# Setup: download the course data into larkfield-experiments/data/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

DATA = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C40/data/"
FILES = {
    "delivery/customers.csv.gz": "c22521f272d36b36fd26234d53d04aac1c8be347a5014ee3e2aeeffcc34a0c19",
    "delivery/truth.json": "6f088f29a112de123bb1da3bac0940a28c5adb706e9f24007686c373c14ab6a4"
}
LOCAL = os.environ.get("NX_DATA_LOCAL")  # authors only: a local copy of the labs data
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("larkfield-experiments").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    if LOCAL:
        shutil.copy(Path(LOCAL) / name, path)
    else:
        download(DATA + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-experiments and run again.")
os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in data/.")

## Setup: helpers

The next cell loads pandas as `pd`, NumPy as `np`, SciPy's `stats` and statsmodels' formula interface as `smf`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `truth(name)` reads a scenario's `truth.json`: the simulator's true effects.

Run the cell. It prints nothing.

In [ ]:
# Helpers: checks that do not show the answer, and the simulator's truth.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
pd.set_option("display.precision", 4)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def truth(name):
    """The simulator's true effects for a scenario (a real test never has this)."""
    return json.loads(open(f"data/{name}/truth.json").read())

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. CUPED on six customers

Six customers, made up so that you can see every value: `x` is what each one spent before the test, and `y` what they spent during it. CUPED computes

adjusted = y − θ × (x − mean of x), with θ = covariance(y, x) / variance(x).

In [ ]:
x = np.array([0, 20, 40, 60, 100, 140], float)
y = np.array([10, 0, 50, 30, 70, 80], float)
theta = np.cov(y, x, ddof=1)[0, 1] / x.var(ddof=1)
adjusted = y - theta * (x - x.mean())
print(f"theta {theta:.3f}")
print("adjusted:", adjusted.round(1))
print(f"mean {y.mean():.1f} -> {adjusted.mean():.1f}; variance {y.var(ddof=1):.1f} -> {adjusted.var(ddof=1):.1f}")

> **Check.** You should see `theta 0.559`, the adjusted values `[43.5 22.4 61.2 30.  47.6 35.3]` and `mean 40.0 -> 40.0; variance 1040.0 -> 190.6`.

The mean does not move. The variance falls by 82%, because `x` predicts `y` well in this small example.

## 2. θ and the correlation at Larkfield

`pre_spend` is each customer's spending in the 8 weeks before the free-delivery test. The coin flip came after it, so free delivery could not change it.

> **Predict.** Will θ for Larkfield be above or below 0.5?

In [ ]:
d = pd.read_csv("data/delivery/customers.csv.gz")
theta = np.cov(d.revenue, d.pre_spend, ddof=1)[0, 1] / d.pre_spend.var(ddof=1)
corr = np.corrcoef(d.revenue, d.pre_spend)[0, 1]
print(f"theta {theta:.3f}, correlation {corr:.3f}, share with no spending before: {(d.pre_spend == 0).mean():.2f}")

> **Check.** You should see `theta 0.271, correlation 0.439, share with no spending before: 0.69`.

Each dollar spent before the test goes with 27 cents spent during it. Most customers spent nothing before, so `pre_spend` predicts only part of the noise.

## 3. The adjusted t-test

Adjust every customer's revenue with the same θ, then run Welch's t-test on the adjusted values. The helper `welch` gives the difference B − A, its 95% interval and the p-value.

In [ ]:
def welch(df, col="revenue"):
    """Difference B - A of a column, its 95% interval and p-value (Welch's t-test)."""
    a, b = df.loc[df.variant == "A", col], df.loc[df.variant == "B", col]
    t = stats.ttest_ind(b, a, equal_var=False)
    ci = t.confidence_interval()
    return round(float(b.mean() - a.mean()), 3), round(float(ci.low), 3), round(float(ci.high), 3), float(f"{t.pvalue:.3g}")

d["adjusted"] = d.revenue - theta * (d.pre_spend - d.pre_spend.mean())
plain, cuped = welch(d), welch(d, "adjusted")
print("without CUPED:", plain)
print("with CUPED:   ", cuped)
print("true effect:  ", truth("delivery")["revenue"]["effect"])

> **Check.** You should see `without CUPED: (1.027, 0.158, 1.895, 0.0205)` and `with CUPED:    (1.043, 0.263, 1.824, 0.00877)`: the same interval as `xp analyse --cuped`. The true effect, 0.6979, is inside both.

## 4. How much narrower?

In [ ]:
w_plain, w_cuped = plain[2] - plain[1], cuped[2] - cuped[1]
print(f"width {w_plain:.3f} without CUPED, {w_cuped:.3f} with it: {w_cuped / w_plain:.2f} times as wide")

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 2))
for row, (name, r) in enumerate([("Without CUPED", plain), ("With CUPED", cuped)]):
    ax.plot([r[1], r[2]], [row, row], color="#0054D8", lw=3)
    ax.plot(r[0], row, "o", color="#0B1A3F")
ax.axvline(0, color="#4A5A76", ls="--"); ax.axvline(truth("delivery")["revenue"]["effect"], color="#B54708", ls=":")
ax.set_yticks([0, 1], ["Without CUPED", "With CUPED"]); ax.set_xlabel("Difference B - A (dollars per customer)")
plt.show()

> **Check.** You should see `width 1.737 without CUPED, 1.561 with it: 0.90 times as wide`.

The customers a test needs grow with the variance, so the same precision needs about a fifth fewer customers.

## 5. Your turn: the share of the variance removed

> **Your turn.** CUPED removes a share of the variance equal to the squared correlation of the metric and the covariate. Compute it for `revenue` and `pre_spend`, and put it in `share_removed`, rounded to 3 decimals.

In [ ]:
share_removed = round(np.corrcoef(d.revenue, d.pre_spend)[0, 1] ** 2, 3)
print(share_removed)

In [ ]:
expect('share_removed', share_removed, 0.193, tolerance=1e-06)

## 6. A covariate from the wrong time

Tomás uses the number of **orders during the test** as the covariate, because it predicts revenue much better. Run it, and compare with the true effect.

In [ ]:
theta_o = np.cov(d.revenue, d.orders, ddof=1)[0, 1] / d.orders.var(ddof=1)
d["wrong"] = d.revenue - theta_o * (d.orders - d.orders.mean())
print(f"variance removed {np.corrcoef(d.revenue, d.orders)[0, 1] ** 2:.1%}")
print("with orders as the covariate:", welch(d, "wrong"))

> **Check.** You should see `variance removed 61.6%` and `with orders as the covariate: (-0.42, -0.958, 0.118, 0.126)`. The true effect is 0.6979.

Free delivery changed the number of orders, so this covariate carries part of the effect, and CUPED removes it. Use only covariates measured before the coin flip.

## Next

You can make a test more precise. The [lesson page](https://learning.nextia-ai.com/courses/experiments/m03/get-more-precise-with-cuped/) has the charts and the knowledge checks that count toward your certificate. Next, we look at segments, and why many segments find false winners.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Segments and different effects](https://learning.nextia-ai.com/courses/experiments/m03/segments-and-different-effects/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/experiments/m03/get-more-precise-with-cuped/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.